In [1]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import random
import numpy as np

In [2]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc1   = nn.Linear(16 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)
        self.pool  = nn.MaxPool2d(2)
        self.relu  = nn.ReLU()
 
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

In [3]:
def set_seed(seed):
    """Makes the run reproducible AND ensures each seed actually differs."""
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
 

In [4]:

def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            output = model(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total  # returns the number, doesn't just print

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [6]:

SEEDS = [0, 42, 123]   # 3 seeds — drop to [0, 42] if time-constrained
 
for seed in SEEDS:
    print(f"\n=== Training seed {seed} ===")
    set_seed(seed)
 
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)  # reshuffled per seed
    model = TinyNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
 
    for epoch in range(5):
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")
 
    baseline_acc = evaluate(model, test_loader)
    print(f"  Seed {seed} baseline accuracy: {baseline_acc:.2f}%")
 
    torch.save(model.state_dict(), f'tinynet_fashion_seed{seed}.pth')
    print(f"  Saved: tinynet_fashion_seed{seed}.pth")


=== Training seed 0 ===
  Epoch 1 | Loss: 0.5217
  Epoch 2 | Loss: 0.3434
  Epoch 3 | Loss: 0.2980
  Epoch 4 | Loss: 0.2688
  Epoch 5 | Loss: 0.2507
  Seed 0 baseline accuracy: 89.62%
  Saved: tinynet_fashion_seed0.pth

=== Training seed 42 ===
  Epoch 1 | Loss: 0.5262
  Epoch 2 | Loss: 0.3487
  Epoch 3 | Loss: 0.3050
  Epoch 4 | Loss: 0.2760
  Epoch 5 | Loss: 0.2578
  Seed 42 baseline accuracy: 89.13%
  Saved: tinynet_fashion_seed42.pth

=== Training seed 123 ===
  Epoch 1 | Loss: 0.5196
  Epoch 2 | Loss: 0.3525
  Epoch 3 | Loss: 0.3089
  Epoch 4 | Loss: 0.2812
  Epoch 5 | Loss: 0.2589
  Seed 123 baseline accuracy: 89.32%
  Saved: tinynet_fashion_seed123.pth


In [7]:

def load_fresh_model(seed):
    model = TinyNet()
    model.load_state_dict(torch.load(f'tinynet_fashion_seed{seed}.pth'))
    model.eval()
    return model
 
def prune_single_layer(model, layer_name, amount):
    layer = getattr(model, layer_name)
    prune.l1_unstructured(layer, name='weight', amount=amount)
    return model
 
# Sweep range — matches your original FashionMNIST full sweep checkpoints
CHECKPOINTS = [0, 10, 20, 30, 40, 45, 50, 55, 60, 65, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80]
LAYERS = ['conv1', 'conv2', 'fc1', 'fc2']
 
results = {}  # results[seed][sparsity] = gap
 
for seed in SEEDS:
    print(f"\n=== Sensitivity sweep, seed {seed} ===")
    baseline_model = load_fresh_model(seed)
    baseline_acc = evaluate(baseline_model, test_loader)
 
    seed_gaps = {}
    for pct in CHECKPOINTS:
        amount = pct / 100
        layer_drops = {}
        for layer_name in LAYERS:
            m = load_fresh_model(seed)          # fresh reload — no carried-forward damage
            m = prune_single_layer(m, layer_name, amount)
            acc = evaluate(m, test_loader)
            drop = baseline_acc - acc
            layer_drops[layer_name] = drop
 
        gap = max(layer_drops.values()) - min(layer_drops.values())
        seed_gaps[pct] = gap
        print(f"  {pct}% sparsity | gap = {gap:.2f}pp | drops = {layer_drops}")
 
    results[seed] = seed_gaps


=== Sensitivity sweep, seed 0 ===
  0% sparsity | gap = 0.00pp | drops = {'conv1': 0.0, 'conv2': 0.0, 'fc1': 0.0, 'fc2': 0.0}
  10% sparsity | gap = 0.09pp | drops = {'conv1': -0.09999999999999432, 'conv2': -0.009999999999990905, 'fc1': -0.04999999999999716, 'fc2': -0.04999999999999716}
  20% sparsity | gap = 0.50pp | drops = {'conv1': 0.3400000000000034, 'conv2': 0.10999999999999943, 'fc1': 0.010000000000005116, 'fc2': -0.1599999999999966}
  30% sparsity | gap = 0.85pp | drops = {'conv1': 0.7700000000000102, 'conv2': 0.13000000000000966, 'fc1': -0.04999999999999716, 'fc2': -0.0799999999999983}
  40% sparsity | gap = 1.92pp | drops = {'conv1': 1.8599999999999994, 'conv2': 0.46000000000000796, 'fc1': -0.060000000000002274, 'fc2': -0.060000000000002274}
  45% sparsity | gap = 2.58pp | drops = {'conv1': 2.5600000000000023, 'conv2': 0.7000000000000028, 'fc1': -0.01999999999999602, 'fc2': 0.45000000000000284}
  50% sparsity | gap = 3.83pp | drops = {'conv1': 3.960000000000008, 'conv2': 0.5

In [8]:
print("\n\n=== SUMMARY: gap by sparsity, across seeds ===")
print(f"{'Sparsity':>10}", end="")
for seed in SEEDS:
    print(f"{'seed'+str(seed):>12}", end="")
print()
 
for pct in CHECKPOINTS:
    print(f"{pct:>9}%", end="")
    for seed in SEEDS:
        print(f"{results[seed][pct]:>12.2f}", end="")
    print()
 
# Save results for later plotting / the reading log
import json
with open('fashionmnist_multiseed_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: fashionmnist_multiseed_results.json")
 



=== SUMMARY: gap by sparsity, across seeds ===
  Sparsity       seed0      seed42     seed123
        0%        0.00        0.00        0.00
       10%        0.09        0.20        0.26
       20%        0.50        0.78        1.57
       30%        0.85        1.79        1.74
       40%        1.92        2.77        2.95
       45%        2.58        4.59        3.16
       50%        3.83        6.38        2.44
       55%        6.04        6.80        6.03
       60%        7.72       10.89        5.19
       65%       10.56       10.01       13.54
       70%       10.21       13.71        7.47
       71%       11.37       17.33        8.60
       72%       16.23       19.12        8.38
       73%       21.84       20.14        8.74
       74%       21.79       19.89       10.59
       75%       12.81       30.09       11.69
       76%       14.19       32.79       12.81
       77%       14.57       32.71       10.28
       78%       16.21       31.25       11.19
       79% 